# Task 3: Dual-Axis Chart — Avg Installs & Revenue, Free vs Paid (Top 3 Categories)

**Requirements:**
- Dual-axis chart comparing **average installs** and **revenue** for **Free vs Paid** apps within the **top 3 app categories**
- Filters:
  - Installs >= 10,000
  - Revenue >= $10,000
  - Android version > 4.0
  - Size > 15 MB
  - Content Rating == Everyone
  - App name length <= 30 characters (incl. spaces/special chars)
- Display rule: only visible between **1 PM – 2 PM IST** (implemented in the dashboard HTML)

### Note on the revenue filter
The dataset has no direct "revenue" column, so we derive it as `Revenue = Price × Installs`. Free apps have `Price = 0`, so their revenue is always **$0** — a revenue filter of "$10,000+" would exclude every free app if applied uniformly, which conflicts with the request to compare free vs paid apps. We therefore apply the **installs / Android version / size / content rating / name-length filters to all apps**, and apply the **$10,000+ revenue filter only to paid apps** (where revenue is a meaningful, non-zero figure). This is a documented assumption, not something stated explicitly in the brief.

In [1]:
import pandas as pd
import numpy as np
import re
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

## 1. Load and clean the data

In [2]:
df = pd.read_csv('googleplaystore.csv')
df = df.drop_duplicates(subset='App', keep='first')

# Drop the known corrupt row (shifted columns -> Category == '1.9')
df = df[~df['Category'].astype(str).str.contains(r'^\d', regex=True, na=False)]

# --- Size -> MB ---
def parse_size(size):
    if pd.isna(size):
        return np.nan
    size = str(size).strip()
    if size == 'Varies with device' or size == '':
        return np.nan
    if size.endswith('M'):
        return float(size[:-1])
    if size.endswith('k') or size.endswith('K'):
        return float(size[:-1]) / 1024.0
    try:
        return float(size)
    except ValueError:
        return np.nan
df['Size_MB'] = df['Size'].apply(parse_size)

# --- Installs -> numeric ---
df['Installs_Num'] = pd.to_numeric(
    df['Installs'].astype(str).str.replace(',', '', regex=False).str.replace('+', '', regex=False),
    errors='coerce'
)

# --- Price -> numeric ---
df['Price_Num'] = pd.to_numeric(df['Price'].astype(str).str.replace('$', '', regex=False), errors='coerce')

# --- Android Ver -> minimum numeric version ---
def parse_android(v):
    if pd.isna(v):
        return np.nan
    m = re.search(r'(\d+(\.\d+)?)', str(v))
    return float(m.group(1)) if m else np.nan
df['Android_Ver_Num'] = df['Android Ver'].apply(parse_android)

# --- Revenue = Price x Installs (0 for Free apps) ---
df['Revenue'] = df['Price_Num'] * df['Installs_Num']
df.loc[df['Type'] == 'Free', 'Revenue'] = 0

# --- App name length ---
df['App_Name_Len'] = df['App'].astype(str).str.len()

df[['App','Category','Type','Installs_Num','Price_Num','Revenue','Android_Ver_Num','Size_MB','Content Rating','App_Name_Len']].head()

,App,Category,Type,Installs_Num,Price_Num,Revenue,Android_Ver_Num,Size_MB,Content Rating,App_Name_Len
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,Free,10000,0.0,0.0,4.0,19.0,Everyone,46
1,Coloring book moana,ART_AND_DESIGN,Free,500000,0.0,0.0,4.0,14.0,Everyone,19
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,Free,5000000,0.0,0.0,4.0,8.7,Everyone,50
3,Sketch - Draw & Paint,ART_AND_DESIGN,Free,50000000,0.0,0.0,4.2,25.0,Teen,21
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,Free,100000,0.0,0.0,4.4,2.8,Everyone,37


## 2. Apply filters

In [3]:
base_mask = (
    (df['Installs_Num'] >= 10_000) &
    (df['Android_Ver_Num'] > 4.0) &
    (df['Size_MB'] > 15) &
    (df['Content Rating'] == 'Everyone') &
    (df['App_Name_Len'] <= 30)
)

paid_mask = base_mask & (df['Type'] == 'Paid') & (df['Revenue'] >= 10_000)
free_mask = base_mask & (df['Type'] == 'Free')

filtered = df[paid_mask | free_mask].copy()
print(f"Rows after filtering: {len(filtered)}  (Free: {(filtered['Type']=='Free').sum()}, Paid: {(filtered['Type']=='Paid').sum()})")

Rows after filtering: 875  (Free: 847, Paid: 28)


## 3. Top 3 categories (by total installs) and aggregation

In [4]:
top3_categories = (
    filtered.groupby('Category')['Installs_Num'].sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)
print('Top 3 categories:', top3_categories)

sub = filtered[filtered['Category'].isin(top3_categories)]

agg = sub.groupby(['Category', 'Type']).agg(
    Avg_Installs=('Installs_Num', 'mean'),
    Avg_Revenue=('Revenue', 'mean'),
    App_Count=('App', 'count')
).reset_index()

# keep category order = top3_categories order
agg['Category'] = pd.Categorical(agg['Category'], categories=top3_categories, ordered=True)
agg = agg.sort_values(['Category', 'Type']).reset_index(drop=True)
agg

Top 3 categories: ['GAME', 'FAMILY', 'TOOLS']


,Category,Type,Avg_Installs,Avg_Revenue,App_Count
0,GAME,Free,3.319490e+07,0.000000,96
1,GAME,Paid,6.714286e+04,276471.428571,7
2,FAMILY,Free,6.423557e+06,0.000000,194
3,FAMILY,Paid,3.616667e+05,597216.666667,6
4,TOOLS,Free,2.148290e+07,0.000000,31
5,TOOLS,Paid,1.000000e+05,449000.000000,1


## 4. Dual-axis chart (Plotly)

Bars (left axis) show **average installs** for Free vs Paid within each category. Markers/lines (right axis) show **average revenue**. Free apps' revenue is always $0 by construction (see note above), which itself is a meaningful part of the free-vs-paid comparison.

In [5]:
free_data = agg[agg['Type'] == 'Free'].set_index('Category').reindex(top3_categories)
paid_data = agg[agg['Type'] == 'Paid'].set_index('Category').reindex(top3_categories)

fig = go.Figure()

# Bars: average installs (left axis)
fig.add_trace(go.Bar(
    x=top3_categories, y=free_data['Avg_Installs'],
    name='Avg Installs (Free)', marker_color='#4C9AFF', yaxis='y1', offsetgroup=0
))
fig.add_trace(go.Bar(
    x=top3_categories, y=paid_data['Avg_Installs'],
    name='Avg Installs (Paid)', marker_color='#1F5FA6', yaxis='y1', offsetgroup=1
))

# Lines: average revenue (right axis)
fig.add_trace(go.Scatter(
    x=top3_categories, y=free_data['Avg_Revenue'],
    name='Avg Revenue (Free)', mode='lines+markers',
    line=dict(color='#FFB454', width=3, dash='dot'), yaxis='y2'
))
fig.add_trace(go.Scatter(
    x=top3_categories, y=paid_data['Avg_Revenue'],
    name='Avg Revenue (Paid)', mode='lines+markers',
    line=dict(color='#FF6B6B', width=3), yaxis='y2'
))

fig.update_layout(
    title='Avg Installs & Revenue — Free vs Paid (Top 3 Categories)',
    barmode='group',
    xaxis=dict(title='Category'),
    yaxis=dict(title='Average Installs', side='left'),
    yaxis2=dict(title='Average Revenue ($)', overlaying='y', side='right', showgrid=False),
    legend=dict(orientation='h', y=1.15, x=0.5, xanchor='center'),
    height=600,
    template='plotly_white',
    margin=dict(t=110, b=60)
)

fig.show()

## 5. Export for the combined dashboard

In [6]:
agg.to_csv('task3_free_vs_paid.csv', index=False)
print('Saved: task3_free_vs_paid.csv')
print('Top 3 categories used:', top3_categories)

Saved: task3_free_vs_paid.csv
Top 3 categories used: ['GAME', 'FAMILY', 'TOOLS']


### Note on the 1PM–2PM IST display rule
Same pattern as Tasks 1 & 2: this is a dashboard-level display rule, implemented with the same IST time-check logic in the combined `dashboard.html`.